In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"

In [ ]:
df_lead_paragraph = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/TimeSentiment.pkl")
df_lead_paragraph = df_lead_paragraph.loc[df_lead_paragraph["Date"]>"2020-05-31"]

In [ ]:
df_lead_paragraph

In [ ]:
aa = df_lead_paragraph["headline"]+" " +df_lead_paragraph["lead_paragraph"]

In [ ]:
df_lead_paragraph["headline"][0]

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[["lead_paragraph"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''
df[['GI_Negative']] = ''
df[['GI_Positive']] = ''
df[['GI_Sentiment']] = ''
df[["GI_Polarity"]] = ''
df[["GI_Subjectivity"]] = ''

In [ ]:
rows_list = []
for date, row in df_lead_paragraph.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['headline']+" "+row['lead_paragraph'])#.encode('ascii','ignore').decode('ascii')
    # sentence = str(preprocessing_text(sentence))

    ss = vader.polarity_scores(sentence)

    Hour = row["Hour"]
    Time = row["Time"]
    Date = row["Date"]
    Compound = ss['compound']
    Neg = ss['neg']
    Pos = ss['pos']
    Neu = ss['neu']
    # Sentiment=0
    if(Pos + Neg):
        Sentiment = (100*((Neg) / (Pos + Neg)))

    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Time":Time, "Date":Date, "Hour": Hour, "Sentiment":Sentiment, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"headline":row["headline"], "lead_paragraph":row["lead_paragraph"]}
    rows_list.append(new_row)

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Time", "Date", "Hour", "Sentiment", "compound", "neg", "neu", "pos", "polarity", "subjectivity", "polarityB","headline","lead_paragraph"])

In [ ]:
# print("Number of Negative words="+str(countNeg))
# print("Number of Positive words="+str(countPos))
# print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/dfTime.pkl")

In [ ]:
df["sentiment"].ewm(10).mean().plot(figsize=(20,10))

In [ ]:
df.index = df.Time

In [ ]:
df